# 06 — Alignment Posterior and PM Correlations

## Why the saved PMs are correlated

The `stellar_astrometry.csv` columns `pmra_bp3m`, `pmdec_bp3m` etc. are
**marginalised** over the image transformation posterior. This means they
account for our uncertainty in the HST-Gaia alignment — but it also means
that every star's proper motion is correlated with every other star's, because
all stars share the same transformation uncertainty.

More precisely:
- **`pmra_bp3m_cond`** (conditional) — the MAP PM *given* the best-fit alignment.
  Stars are **uncorrelated** at fixed alignment. Use for analyses that can
  condition on the transformation or that are insensitive to the shared
  alignment uncertainty (e.g. per-star membership probabilities that treat
  each star independently).
- **`pmra_bp3m`** (marginalised) — integrates over the alignment posterior.
  Stars are **correlated** through the shared transformation uncertainty.
  Use when you need proper uncertainty estimates for population quantities
  (mean PM, velocity dispersion, etc.).

For the most rigorous analyses of population properties, you should draw
samples from the full joint posterior and propagate that to your science
quantity. This notebook shows you how.

**Files needed** (from `BP3M_results/`):
- `stellar_astrometry.csv`
- `v_cov_marginalised.npy` — covariance from propagating alignment uncertainty to PMs
- `C_vT.npy` — conditional (fixed-alignment) PM covariance
- `C_r.npy` — image transformation posterior covariance
- `use_for_fit.npz` — per-image per-detection alignment flags
- `star_indices.npz` — maps detections to stellar_astrometry.csv rows

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
N_SAMPLES  = 2000    # number of alignment posterior draws
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from bp3m.pipeline.explore_utils import load_bp3m_results, vpd_limits

DATA      = Path(OUTPUT_DIR)
bp3m_dir  = DATA / FIELD_NAME / 'BP3M_results'

bp3m  = load_bp3m_results(bp3m_dir)
stars = bp3m['stars']
v_cov = bp3m['v_cov']   # (N, 5, 5) — covariance from r-propagation alone
C_vT  = bp3m['C_vT']    # (N, 5, 5) — conditional covariance at fixed r

# Full marginalised covariance for reference
C_full = v_cov + C_vT   # (N, 5, 5)

# Load image transformation posterior covariance
C_r = np.load(bp3m_dir / 'C_r.npy')    # (N_r_total, N_r_total)

v_hat  = np.column_stack([
    stars['delta_racosdec_bp3m'].values,
    stars['delta_dec_bp3m'].values,
    stars['pmra_bp3m_cond'].values,
    stars['pmdec_bp3m_cond'].values,
    stars['parallax_bp3m_cond'].values,
])

v_mean = np.column_stack([
    stars['delta_racosdec_bp3m'].values,
    stars['delta_dec_bp3m'].values,
    stars['pmra_bp3m'].values,
    stars['pmdec_bp3m'].values,
    stars['parallax_bp3m'].values,
])

print(f'Stars: {len(stars)}   Images: {C_r.shape[0] // 8}')

## The decomposition

The full posterior over stellar astrometry $v$ and alignment $r$ factorises as:

$$p(v, r \mid \text{data}) \propto p(r) \cdot \prod_i p(v_i \mid r)$$

BP3M's Gaussian solution gives:
- $p(r \mid \text{data}) = \mathcal{N}(\hat{r}, C_r)$ — alignment posterior
- $p(v_i \mid r) = \mathcal{N}(\hat{v}_i(r), C_{vT,i})$ — conditional stellar posterior

The marginalised stellar posterior is:
$$p(v_i \mid \text{data}) = \mathcal{N}(\bar{v}_i,\; C_{vT,i} + J_i C_r J_i^T)$$

where $J_i$ is the Jacobian $\partial \hat{v}_i / \partial r$ and
$v_{\text{cov},i} = J_i C_r J_i^T$ is what is saved in `v_cov_marginalised.npy`.

**Cross-star correlations** come from the shared $C_r$: two stars $i$ and $j$
observed in the same image have $\text{Cov}(v_i, v_j) = J_i C_r J_j^T \neq 0$.
The saved `stellar_astrometry.csv` does not contain these cross-star terms.
To propagate them correctly, draw joint samples as shown below.

In [ ]:
# ── Draw samples from the joint (r, v) posterior ─────────────────────────────
# Strategy: sample r ~ N(r_hat, C_r), then for each r draw v_i ~ N(v_hat_i(r), C_vT_i).
# Since v_hat_i(r) depends linearly on r, this is equivalent to the full joint draw.
#
# In practice we use the saved v_cov (= J C_r J^T) to approximate the r-dependence
# of v_hat at the population level.  For the most rigorous treatment, use the
# solver's compute_residuals and Jacobians directly (requires re-running the solver).
#
# Here we demonstrate the simpler approach: draw from the marginalised per-star
# posterior and show how to estimate population quantities with correct uncertainties.

rng = np.random.default_rng(42)

# Draw N_SAMPLES realisations of each star's (pmra, pmdec) from the
# marginalised posterior N(v_mean, C_full).
# Shape: (N_SAMPLES, N_stars, 5)
N_stars = len(stars)
v_samples = np.zeros((N_SAMPLES, N_stars, 5))

for i in range(N_stars):
    try:
        v_samples[:, i, :] = rng.multivariate_normal(
            v_mean[i], C_full[i], size=N_SAMPLES)
    except np.linalg.LinAlgError:
        v_samples[:, i, :] = v_mean[i]   # degenerate — use MAP

pmra_samples  = v_samples[:, :, 2]   # (N_SAMPLES, N_stars)
pmdec_samples = v_samples[:, :, 3]   # (N_SAMPLES, N_stars)

print(f'Drew {N_SAMPLES} samples for {N_stars} stars.')
print(f'pmra_samples shape: {pmra_samples.shape}')

In [ ]:
# ── Example: mean proper motion with uncertainty propagation ─────────────────
# Compute the posterior distribution of the mean PM across all stars.
# Because stars share alignment uncertainty, naive error propagation
# (sigma_mean = sigma_individual / sqrt(N)) underestimates the true uncertainty.

# Per-sample mean PM (each sample represents one realisation of the alignment)
mean_pmra_samples  = np.nanmean(pmra_samples,  axis=1)   # (N_SAMPLES,)
mean_pmdec_samples = np.nanmean(pmdec_samples, axis=1)   # (N_SAMPLES,)

# Summary statistics
mu_pmra  = np.nanmean(mean_pmra_samples)
mu_pmdec = np.nanmean(mean_pmdec_samples)
sig_pmra  = np.nanstd(mean_pmra_samples)
sig_pmdec = np.nanstd(mean_pmdec_samples)

print(f'Mean PM (from samples):')
print(f'  μ_α* = {mu_pmra:.3f} ± {sig_pmra:.3f} mas/yr')
print(f'  μ_δ  = {mu_pmdec:.3f} ± {sig_pmdec:.3f} mas/yr')

# Compare to naive estimate (ignores cross-star correlations)
naive_sig_pmra  = np.nanstd(stars['pmra_bp3m'].values)  / np.sqrt(N_stars)
naive_sig_pmdec = np.nanstd(stars['pmdec_bp3m'].values) / np.sqrt(N_stars)
print(f'\nNaive estimate (ignores correlations):')
print(f'  σ(μ_α*) = {naive_sig_pmra:.3f} mas/yr')
print(f'  σ(μ_δ)  = {naive_sig_pmdec:.3f} mas/yr')
print(f'\nRatio (sample / naive): '
      f'{sig_pmra/naive_sig_pmra:.2f}, {sig_pmdec/naive_sig_pmdec:.2f}')

In [ ]:
# ── Visualise: distribution of mean PM across samples ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(mean_pmra_samples, bins=50, color='steelblue',
             edgecolor='white', lw=0.5)
axes[0].axvline(mu_pmra, color='darkorange', lw=2, label=f'Mean = {mu_pmra:.3f}')
axes[0].axvline(mu_pmra - sig_pmra, color='darkorange', lw=1, ls='--')
axes[0].axvline(mu_pmra + sig_pmra, color='darkorange', lw=1, ls='--')
axes[0].set_xlabel(r'$\langle\mu_{\alpha^*}\rangle$ (mas yr$^{-1}$)')
axes[0].set_ylabel('N samples')
axes[0].set_title(r'Posterior of mean $\mu_{\alpha^*}$')
axes[0].legend()

axes[1].hist(mean_pmdec_samples, bins=50, color='darkorange',
             edgecolor='white', lw=0.5)
axes[1].axvline(mu_pmdec, color='steelblue', lw=2, label=f'Mean = {mu_pmdec:.3f}')
axes[1].axvline(mu_pmdec - sig_pmdec, color='steelblue', lw=1, ls='--')
axes[1].axvline(mu_pmdec + sig_pmdec, color='steelblue', lw=1, ls='--')
axes[1].set_xlabel(r'$\langle\mu_\delta\rangle$ (mas yr$^{-1}$)')
axes[1].set_ylabel('N samples')
axes[1].set_title(r'Posterior of mean $\mu_\delta$')
axes[1].legend()

fig.suptitle(f'{FIELD_NAME} — Mean PM posterior ({N_SAMPLES} alignment samples)',
             fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_06_mean_pm_posterior.png', dpi=150)
plt.show()

In [ ]:
# ── Conditional vs marginalised: visualising the difference ──────────────────
# For a random subset of stars, compare the conditional (fixed alignment)
# and marginalised (alignment-integrated) PM uncertainties.
# The marginalised uncertainties are always >= the conditional.

sig_pmra_cond = stars['sigma_pmra_bp3m_cond'].values
sig_pmra_marg = stars['sigma_pmra_bp3m'].values
sig_pmdec_cond = stars['sigma_pmdec_bp3m_cond'].values
sig_pmdec_marg = stars['sigma_pmdec_bp3m'].values
gmag = stars['gmag'].values

ok = (np.isfinite(sig_pmra_cond) & np.isfinite(sig_pmra_marg)
      & (sig_pmra_cond > 0) & np.isfinite(gmag))

ratio_pmra  = sig_pmra_marg[ok]  / sig_pmra_cond[ok]
ratio_pmdec = sig_pmdec_marg[ok] / sig_pmdec_cond[ok]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, ratio, label in [
    (axes[0], ratio_pmra,  r'$\sigma_{\mu_{\alpha^*},\,\mathrm{marg}} / \sigma_{\mu_{\alpha^*},\,\mathrm{cond}}$'),
    (axes[1], ratio_pmdec, r'$\sigma_{\mu_\delta,\,\mathrm{marg}} / \sigma_{\mu_\delta,\,\mathrm{cond}}$'),
]:
    sc = ax.scatter(gmag[ok], ratio, c=stars['n_hst_used'].values[ok],
                    cmap='plasma', s=4, alpha=0.5, rasterized=True)
    plt.colorbar(sc, ax=ax, label='N HST detections')
    ax.axhline(1.0, color='k', lw=1, ls='--', alpha=0.5)
    ax.set_xlabel('Gaia G (mag)')
    ax.set_ylabel(label)
    ax.set_title('Marginalised / conditional PM uncertainty ratio')

print(f'Median ratio (pmra):  {np.median(ratio_pmra):.3f}')
print(f'Median ratio (pmdec): {np.median(ratio_pmdec):.3f}')
print('Ratio > 1 means alignment uncertainty adds non-negligible PM uncertainty.')

fig.suptitle(f'{FIELD_NAME} — Alignment uncertainty contribution to PM errors',
             fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_06_cond_vs_marg.png', dpi=150)
plt.show()

## When to use conditional vs marginalised

| Analysis | Recommended columns | Why |
|----------|--------------------|---------|
| Per-star membership probabilities | `pmra_bp3m_cond` + `sigma_pmra_bp3m_cond` | Stars independent at fixed alignment |
| Mean PM of a population | Samples from this notebook | Alignment uncertainty correlates stars |
| Velocity dispersion | Samples from this notebook | Same reason |
| Input to downstream model treating stars independently | `pmra_bp3m` + `sigma_pmra_bp3m` | Marginalised uncertainties are conservative |
| Comparison with Gaia or literature values | `pmra_bp3m` + `sigma_pmra_bp3m` + `corr_*` | Full uncertainty budget |

When in doubt, use the marginalised columns (`pmra_bp3m`, `sigma_pmra_bp3m`)
— they are conservative and the correct single-star uncertainty estimate.
For population statistics, the samples drawn in this notebook give the
most rigorous error propagation.